<a href="https://colab.research.google.com/github/tobp03/word2vec-implementation/blob/main/wod2vec.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Word2Vec Skip-Gram with Negative sampling
This notebook Skip-Gram Word2Vec with Negative Sampling. More explanations can be found [here](https://tobypurbojo.com/notes/Understanding%20Word2Vec%20324df4fe-71d8-80a3-bf8d-f09ff99a71c0).

Main goal: Create word embeddings from raw text using Numpy.


## 1. Download dataset and import libraries
Dataset used Text8 dataset, a cleaned subset of the English Wikipedia corpus. contains about 100 million characters of plain text extracted from Wikipedia. All text is lowercased, bold text and punctuation, numbers, and special characters are removed.  

Source: http://mattmahoney.net/dc/text8.zip



In [1]:
from tqdm import tqdm
import numpy as np

#Download dataset
!wget http://mattmahoney.net/dc/text8.zip
!unzip text8.zip

--2026-03-15 20:49:21--  http://mattmahoney.net/dc/text8.zip
Resolving mattmahoney.net (mattmahoney.net)... 20.119.76.151
Connecting to mattmahoney.net (mattmahoney.net)|20.119.76.151|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 31344016 (30M) [application/zip]
Saving to: ‘text8.zip.3’

text8.zip.3         100%[===================>]  29.89M  --.-KB/s    in 0.1s    

2026-03-15 20:49:21 (311 MB/s) - ‘text8.zip.3’ saved [31344016/31344016]

Archive:  text8.zip
replace text8? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: text8                   


## 2. Text processing
Steps:
1. Read the dataset
2. Split into tokens
3. Build the vocabulary
4. Map words to vocab index

In [2]:
def load_text(path):
    with open(path, "r") as f:
        return f.read()

def tokenize(text, token_size=None):
    tokens = text.split()
    if token_size:
        tokens = tokens[:token_size]
    return tokens

def dataset_stats(text, tokens, vocab):
    print("Total characters:", len(text))
    print("Total tokens:", len(tokens))
    print("Vocabulary size:", len(vocab))

In [3]:
text = load_text("text8")
tokens = tokenize(text, 100_000)
vocab = sorted(set(tokens))

dataset_stats(text, tokens, vocab)

word_to_idx = {word: i for i, word in enumerate(vocab)}
idx_to_word = {i: word for word, i in word_to_idx.items()}

#Convert tokens to vocabulary index
token_ids = [word_to_idx[word] for word in tokens]

Total characters: 100000000
Total tokens: 100000
Vocabulary size: 12023


## 3. Skip-Gram pairs
Skip-gram requires the **center word** as input and the **context word** as output.

For a window size of 2, the sentence:

*`The cat sits on the mat.`*

Pairs (center,context):
- (cat, The)
- (cat, sits)
- (cat, on)
- (sits, The)
- (sits, cat)
- (sits, on)
- (sits, the)
- And so on.



In [4]:
def generate_skipgram_pairs(token_ids, window_size=2):
    pairs = []
    n_tokens = len(token_ids)

    for i, center in enumerate(token_ids):

        # window max boundaries
        left = max(0, i - window_size)
        right = min(n_tokens, i + window_size + 1)

        # collect context words
        for j in range(left, right):
            if j == i:
                continue

            context = token_ids[j]
            pairs.append((center, context))

    return pairs

pairs = generate_skipgram_pairs(token_ids, window_size=2)
print("Total training pairs:", len(pairs))
print("First 10 pairs:", pairs[:10])

Total training pairs: 399994
First 10 pairs: [(538, 7705), (538, 819), (7705, 538), (7705, 819), (7705, 0), (819, 538), (819, 7705), (819, 0), (819, 10811), (0, 7705)]


## 4. Training Loop

Define:
- $V$ : vocabulary size
- $D$ : embedding dimension
- $K$ : number of negative samples


Embedding matrices:
\begin{align*}
W_1 &\in \mathbb{R}^{D \times V}, &
W_2 &\in \mathbb{R}^{D \times V}
\end{align*}

For each epoch, loop over all pairs $(c,o)$ where:
- $c$ = center word
- $o$ = context word

Extract the embeddings:
\begin{align*}
v_c &= W_1[:,c] \\
v_o &= W_2[:,o]
\end{align*}

Forward Pass:

\begin{align*}
s &= v_c^\top v_o \\
\sigma(s) &= \frac{1}{1 + e^{-s}}
\end{align*}

Loss (positive and negative parts):

\begin{align*}
L =
-\left(
\underbrace{\log \sigma(v_c^\top v_o)}_{\text{positive}}
+
\sum_{k=1}^{K}
\underbrace{\log \sigma(-v_c^\top v_{n_k})}_{\text{negative}}
\right)
\end{align*}

Gradient Updates

For the positive pair:

\begin{align*}
\frac{\partial L}{\partial v_c}
&=
(\sigma(v_c^\top v_o) - 1)\, v_o
\end{align*}

\begin{align*}
\frac{\partial L}{\partial v_o}
&=
(\sigma(v_c^\top v_o) - 1)\, v_c
\end{align*}

For negative samples:

\begin{align*}
\frac{\partial L}{\partial v_c}
&=
\sum_{k=1}^{K}
\sigma(v_c^\top v_{n_k})\, v_{n_k}
\end{align*}

\begin{align*}
\frac{\partial L}{\partial v_{n_k}}
&=
\sigma(v_c^\top v_{n_k})\, v_c
\end{align*}
Remember that $v_c$ and $v_o$ is sliced from the weight matrix.

In [5]:
vocab_size = len(vocab)
embedding_dim = 50
learning_rate = 0.05
epochs = 6
neg_samples = 5

W1 = np.random.randn(vocab_size, embedding_dim) * 0.01
W2 = np.random.randn(vocab_size, embedding_dim) * 0.01

# Sigmoid
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# Training loop
for epoch in range(epochs):

    total_loss = 0
    # np.random.shuffle(pairs)  # shuffle training data
    for center, context in tqdm(pairs, desc=f"Epoch {epoch+1}"):

        v_c = W1[center] #(D X 1)

        # Positive pair
        v_o = W2[context] #(D X 1)

        score_pos = sigmoid(v_c @ v_o)
        loss = -np.log(score_pos + 1e-9)

        grad_pos = score_pos - 1

        W1[center] -= learning_rate * grad_pos * v_o
        W2[context] -= learning_rate * grad_pos * v_c

        # Negative samples
        neg_words = np.random.randint(0, vocab_size, size=neg_samples)

        # avoid accidentally sampling the positive word
        neg_words = neg_words[neg_words != context]

        v_n = W2[neg_words] # neg_samples x D

        scores_neg = sigmoid(v_n @ v_c) # (NxD)(Dx1) -> (N x 1)

        loss -= np.sum(np.log(1 - scores_neg + 1e-9))

        grad_neg = scores_neg

        W1[center] -= learning_rate * np.sum(grad_neg[:, None] * v_n, axis=0)
        W2[neg_words] -= learning_rate * grad_neg[:, None] * v_c

        total_loss += loss

    print(f"Epoch {epoch+1} loss: {total_loss:.4f}")

Epoch 1: 100%|██████████| 399994/399994 [00:44<00:00, 9044.84it/s] 


Epoch 1 loss: 866677.1833


Epoch 2: 100%|██████████| 399994/399994 [00:43<00:00, 9113.27it/s] 


Epoch 2 loss: 597859.5899


Epoch 3: 100%|██████████| 399994/399994 [00:43<00:00, 9178.58it/s] 


...

Epoch 4 loss: 521009.5407


Epoch 5: 100%|██████████| 399994/399994 [00:43<00:00, 9254.14it/s] 


Epoch 5 loss: 495445.0114


Epoch 6: 100%|██████████| 399994/399994 [00:43<00:00, 9155.77it/s] 

Epoch 6 loss: 471911.0101


## 5. Cosine similarity

\begin{align}
\text{cosine}(a,b)
&=
\frac{a \cdot b}{\|a\| \, \|b\|} \\
\end{align}

The cosine similarity ranges from -1 to 1:
- 1 : vectors are very similar  
- 0 : unrelated  
- -1 : opposite directions  



In [6]:
from numpy.linalg import norm

def cosine(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

def similarity(w1, w2):
    v1 = W1[word_to_idx[w1]]
    v2 = W1[word_to_idx[w2]]
    return cosine(v1, v2)

print("man vs woman:", similarity("man", "woman"))
print("cat vs dog:", similarity("cat", "dog"))
print("house vs light:", similarity("house", "light"))


man vs woman: 0.7934568369793724
cat vs dog: 0.7230085531820786
house vs light: 0.5622167908843512
